In [1]:
import pandas as pd
import numpy as np
import os
import pyotp
import datetime as dt
from datetime import datetime
import json
import time
import requests

from kiteconnect import KiteConnect
from kiteconnect import KiteTicker
from dotenv import load_dotenv
load_dotenv()  # Load credentials from .env file (not committed to version control)
creds = {
    'user_id': os.getenv('ZERODHA_USER_ID'),
    'password': os.getenv('ZERODHA_PASSWORD'),
    'totp_key': os.getenv('ZERODHA_TOTP_KEY'),
    'api_key': os.getenv('ZERODHA_API_KEY'),
    'api_secret': os.getenv('ZERODHA_API_SECRET')
}
base_url = "https://kite.zerodha.com"
login_url = "https://kite.zerodha.com/api/login"
twofa_url = "https://kite.zerodha.com/api/twofa"
instruments_url = "https://api.kite.trade/instruments"

session = requests.Session()
response = session.post(login_url,data={'user_id':creds['user_id'],'password':creds['password']})
request_id = json.loads(response.text)['data']['request_id']
twofa_pin = pyotp.TOTP(creds['totp_key']).now()
# twofa_pin =149166
response_1 = session.post(twofa_url,data={'user_id':creds['user_id'],'request_id':request_id,'twofa_value':twofa_pin,'twofa_type':'totp'})
kite = KiteConnect(api_key=creds['api_key'])
kite_url = kite.login_url()
try:
    session.get(kite_url)
except Exception as e:
    e_msg = str(e)
#print(e_msg)
request_token = e_msg.split('request_token=')[1].split(' ')[0].split('&action')[0]
print('Successful Login with Request Token:{}'.format(request_token))
data = kite.generate_session(request_token,creds['api_secret'])
access_token =  data['access_token']

kite.set_access_token(access_token)
#Get last traded prices for Nifty and BankNifty
# Initialize Kite Ticker
kws = KiteTicker(creds['api_key'], access_token)



Successful Login with Request Token:rglnEFm2r4bdtwLLoERq4zDMrzcO1bP5


In [2]:
#Get all trading instruments
instrument=kite.instruments()
print(instrument[0])

{'instrument_token': 262676230, 'exchange_token': '1026079', 'tradingsymbol': 'EURINR24AUGFUT', 'name': 'EURINR', 'last_price': 0.0, 'expiry': datetime.date(2024, 8, 28), 'strike': 0.0, 'tick_size': 0.0025, 'lot_size': 1, 'instrument_type': 'FUT', 'segment': 'BCD-FUT', 'exchange': 'BCD'}


In [3]:
# importing module 
import pyspark 

# importing sparksession from 
# pyspark.sql module 
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import datetime
from pyspark.sql.functions import desc, asc, like

# creating sparksession and giving 
# an app name 
spark = SparkSession.builder.appName('sparkdf').getOrCreate() 

# Define the schema for the DataFrame
schema = StructType([
    StructField("instrument_token", IntegerType(), nullable=False),
    StructField("exchange_token", StringType(), nullable=False),
    StructField("tradingsymbol", StringType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("last_price", DoubleType(), nullable=False),
    StructField("expiry", DateType(), nullable=False),
    StructField("strike", DoubleType(), nullable=False),
    StructField("tick_size", DoubleType(), nullable=False),
    StructField("lot_size", IntegerType(), nullable=False),
    StructField("instrument_type", StringType(), nullable=False),
    StructField("segment", StringType(), nullable=False),
    StructField("exchange", StringType(), nullable=False)
])

#Select required instruments only for Nifty and BankNifty
Banknifty_Options = [instrument for instrument in instrument if instrument['exchange'] == 'NFO' and instrument['segment'] == 'NFO-OPT' and instrument['name'] == 'BANKNIFTY']
Nifty_Options = [instrument for instrument in instrument if instrument['exchange'] == 'NFO' and instrument['segment'] == 'NFO-OPT' and instrument['name'] == 'NIFTY']
Banknifty_Futures = [instrument for instrument in instrument if instrument['exchange'] == 'NFO' and instrument['segment'] == 'NFO-FUT' and instrument['name'] == 'BANKNIFTY']
Nifty_Futures = [instrument for instrument in instrument if instrument['exchange'] == 'NFO' and instrument['segment'] == 'NFO-FUT' and instrument['name'] == 'NIFTY']

# creating a dataframe 
Banknifty_Options_DF = spark.createDataFrame(Banknifty_Options, schema=schema)
Nifty_Options_DF = spark.createDataFrame(Nifty_Options, schema=schema)
Banknifty_Futures_DF = spark.createDataFrame(Banknifty_Futures, schema=schema)
Nifty_Futures_DF = spark.createDataFrame(Nifty_Futures, schema=schema)
#sorting by expiry
Banknifty_Options_DF = Banknifty_Options_DF.orderBy(asc("expiry"))
Nifty_Options_DF = Nifty_Options_DF.orderBy(asc("expiry"))
Banknifty_Futures_DF = Banknifty_Futures_DF.orderBy(asc("expiry"))
Nifty_Futures_DF = Nifty_Futures_DF.orderBy(asc("expiry"))




Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/08 09:58:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/08 09:58:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
# display dataframe 
# Banknifty_Options_DF.show(truncate=False)
# Banknifty_Options_DF.select("expiry").distinct().show()
Nifty_Options_DF.show(truncate=False)
Nifty_Options_DF.select("expiry").distinct().show()

+----------------+--------------+-----------------+-----+----------+----------+-------+---------+--------+---------------+-------+--------+
|instrument_token|exchange_token|tradingsymbol    |name |last_price|expiry    |strike |tick_size|lot_size|instrument_type|segment|exchange|
+----------------+--------------+-----------------+-----+----------+----------+-------+---------+--------+---------------+-------+--------+
|13253378        |51771         |NIFTY2480824300CE|NIFTY|0.0       |2024-08-08|24300.0|0.05     |25      |CE             |NFO-OPT|NFO     |
|13253890        |51773         |NIFTY2480824300PE|NIFTY|0.0       |2024-08-08|24300.0|0.05     |25      |PE             |NFO-OPT|NFO     |
|13252610        |51768         |NIFTY2480824250CE|NIFTY|0.0       |2024-08-08|24250.0|0.05     |25      |CE             |NFO-OPT|NFO     |
|13252866        |51769         |NIFTY2480824250PE|NIFTY|0.0       |2024-08-08|24250.0|0.05     |25      |PE             |NFO-OPT|NFO     |
|13254146        |51

In [5]:
#Writing intruments to files

Nifty_Options_DF.coalesce(1).write.format('Delta').mode("Overwrite").parquet("DataFiles/Instruments/Nifty_Options")
Banknifty_Options_DF.coalesce(1).write.format('Delta').mode("Overwrite").parquet("DataFiles/Instruments/Banknifty_Options")
Nifty_Futures_DF.coalesce(1).write.format('Delta').mode("Overwrite").parquet("DataFiles/Instruments/Nifty_Futures")
Banknifty_Futures_DF.coalesce(1).write.format('Delta').mode("Overwrite").parquet("DataFiles/Instruments/Banknifty_Futures")


In [6]:
from pyspark.sql.functions import min
# Current Weekly and monthly expiry
Banknifty_current_week_expiry   = Banknifty_Options_DF.agg(min("expiry")).collect()[0][0]
Banknifty_current_month_expiry  = Banknifty_Futures_DF.agg(min("expiry")).collect()[0][0]
Nifty_current_week_expiry       = Nifty_Options_DF.agg(min("expiry")).collect()[0][0]
Nifty_current_month_expiry      = Nifty_Futures_DF.agg(min("expiry")).collect()[0][0]
Banknifty_next_week_expiry      = Banknifty_Options_DF.filter(Banknifty_Options_DF['expiry'] > Banknifty_current_week_expiry).agg(min("expiry")).collect()[0][0]
Banknifty_next_month_expiry     = Banknifty_Futures_DF.filter(Banknifty_Futures_DF['expiry'] > Banknifty_current_month_expiry).agg(min("expiry")).collect()[0][0]
Nifty_next_week_expiry          = Nifty_Options_DF.filter(Nifty_Options_DF['expiry'] > Nifty_current_week_expiry).agg(min("expiry")).collect()[0][0]
Nifty_next_month_expiry         = Nifty_Futures_DF.filter(Nifty_Futures_DF['expiry'] > Nifty_current_month_expiry).agg(min("expiry")).collect()[0][0]

#create list for expiry dates
Data1 = [('BANKNIFTY', Banknifty_current_week_expiry, Banknifty_current_month_expiry, Banknifty_next_week_expiry, Banknifty_next_month_expiry),
         ('NIFTY', Nifty_current_week_expiry, Nifty_current_month_expiry, Nifty_next_week_expiry, Nifty_next_month_expiry)]

# Define the schema for the DataFrame
schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("current_week_expiry", DateType(), nullable=False),
    StructField("current_month_expiry", DateType(), nullable=False),
    StructField("next_week_expiry", DateType(), nullable=False),
    StructField("next_month_expiry", DateType(), nullable=False)
])
#Writing intruments to files
spark.createDataFrame(Data1, schema=schema).coalesce(1).write.format('Delta').mode("Overwrite").parquet("DataFiles/Instruments/Nifty_Banknifty_Expiries")


In [7]:
def round_to_hundred(num):
    if num % 100 >= 50:
        return num + (100 - num % 100)
    else:
        return num - (num % 100)
    
def round_to_fifty(num):
    if num % 50 >= 25:
        return num + (50 - num % 50)
    else:
        return num - (num % 50)

In [8]:
# #Get Instrument tokens for Nifty and BankNifty
# Banknifty_instrument_token = [instrument for instrument in instrument if instrument['name'] == 'NIFTY BANK'][0]['instrument_token']
# Nifty_instrument_token = [instrument for instrument in instrument if instrument['name'] == 'NIFTY'][0]['instrument_token']
# #get Last traded prices
# Banknifty_ltp = (kite.ltp(Banknifty_instrument_token)[str(Banknifty_instrument_token)]["last_price"])
# Nifty_ltp = (kite.ltp(Nifty_instrument_token)[str(Nifty_instrument_token)]["last_price"])

# #get ATM Strikes
# Banknifty_ATM_Strikes = round_to_hundred(Banknifty_ltp)
# Nifty_ATM_Strikes=round_to_fifty(Nifty_ltp)
# print(Nifty_ltp, Nifty_ATM_Strikes, Banknifty_ltp, Banknifty_ATM_Strikes)

In [9]:
round_to_fifty(22976)

23000

24/08/08 09:58:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
